In [1]:
import os
import vitis
import shutil

In [2]:
# 1. Initialize the headless Vitis server engine
client = vitis.create_client()

Vitis Server started on port '42735'.
Vitis Server for Embedded Edition. Only embedded component related functions are supported.


In [3]:
home = os.path.expanduser('~')
# 2. Assign your workspace path (Change to your desired directory)
ws_path = os.path.join(home,"vitis_ws")
client.set_workspace(path=ws_path)

True

In [4]:
# check and create link

def create_link(target, dest):
    # Check if the source file actually exists
    if not os.path.exists(target):
        raise FileNotFoundError(f"Error: Target source file not found at: {target}")

    # Extract and check the destination's parent folder
    dest_dir = os.path.dirname(dest)

    if not os.path.exists(dest_dir):
        print(f"Destination folder missing. Creating directory: {dest_dir}")
        # os.makedirs creates the directory and any missing parent folders
        os.makedirs(dest_dir, exist_ok=True)
    
    # Check if a file or link already exists at the destination
    if os.path.islink(dest) or os.path.exists(dest):
        # Always remove old references first to avoid "FileExistsError"
        if os.path.isdir(dest) and not os.path.islink(dest):
            os.rmdir(dest)
        else:
            os.remove(dest)

    # Create the symlink
    os.symlink(target, dest)

    print(f"Created link: {dest} -> {target}")

In [5]:
# Name definitions
platform_name = "zynq_zed"
app_names = ["zynq_zed_leds_app", "zynq_zed_fib_app"]

# Extract textual names from the dictionary array
existing_components = client.list_components()
existing_names = [comp['name'] for comp in existing_components]

for app_name in app_names:
    
    src_files = ["platform.h", "platform.c", f"{app_name}.c"]
    
    # Handle Application Component checking/creation
    if app_name in existing_names:
        print(f"Application component '{app_name}' already exists. Loading...")
        app_comp = client.get_component(name=app_name)
        
        for src_file in src_files:
            target = os.path.join(os.getcwd(),"apps_src", src_file)
            dest = os.path.join(ws_path, app_name, "src", src_file)
            create_link(target, dest)
    else:
        print(f"Application component '{app_name}' not found. Initializing creation workflow...")
    
        # 4. Construct the path to your platform's exported .xpfm file
        # Vitis generates this inside the platform export directory structure
        platform_xpfm_path = os.path.join(ws_path, platform_name, "export", platform_name, f"{platform_name}.xpfm")
    
        # 5. Core API method to spin up the component
        app_comp = client.create_app_component(
            name=app_name,
            platform=platform_xpfm_path,
            domain="standalone_domain",     # Match the exact domain name given in your zynq_zed platform bsp
            template="empty_application"          # Core options include: "hello_world" or "empty_application"
        )
        print(f"Application component '{app_name}' successfully created.")
    
        # Populate source directories with external VHDL/C code hooks
        app_comp.import_files(
            from_loc="apps_src/",
            files=src_files,
            dest_dir_in_cmp="src"
        )
    
    # Clean compilation
    app_comp.clean()
    # 6. Execute compilation
    app_comp.build()

Application component 'zynq_zed_leds_app' already exists. Loading...
Created link: /home/maps/vitis_ws/zynq_zed_leds_app/src/platform.h -> /home/maps/ycorrales/Zedboard/vitis/apps_src/platform.h
Created link: /home/maps/vitis_ws/zynq_zed_leds_app/src/platform.c -> /home/maps/ycorrales/Zedboard/vitis/apps_src/platform.c
Created link: /home/maps/vitis_ws/zynq_zed_leds_app/src/zynq_zed_leds_app.c -> /home/maps/ycorrales/Zedboard/vitis/apps_src/zynq_zed_leds_app.c
/bin/gmake  -f CMakeFiles/Makefile2 clean
gmake[1]: Entering directory '/home/maps/vitis_ws/zynq_zed_leds_app/build'
/bin/gmake  -f CMakeFiles/zynq_zed_leds_app.elf.dir/build.make CMakeFiles/zynq_zed_leds_app.elf.dir/clean
gmake[2]: Entering directory '/home/maps/vitis_ws/zynq_zed_leds_app/build'
/home/maps/Xilinx/Vitis/2024.1/tps/lnx64/cmake-3.24.2/bin/cmake -P CMakeFiles/zynq_zed_leds_app.elf.dir/cmake_clean.cmake
gmake[2]: Leaving directory '/home/maps/vitis_ws/zynq_zed_leds_app/build'
gmake[1]: Leaving directory '/home/maps/v

[100%] Linking C executable zynq_zed_fib_app.elf
/home/maps/Xilinx/Vitis/2024.1/tps/lnx64/cmake-3.24.2/bin/cmake -E cmake_link_script CMakeFiles/zynq_zed_fib_app.elf.dir/link.txt --verbose=1
/home/maps/Xilinx/Vitis/2024.1/gnu/aarch32/lin/gcc-arm-none-eabi/bin/arm-none-eabi-gcc  -O2 -DSDT -mcpu=cortex-a9 -mfpu=vfpv3 -mfloat-abi=hard  -MMD -MP -specs=/home/maps/vitis_ws/zynq_zed/export/zynq_zed/sw/standalone_ps7_cortexa9_0/Xilinx.spec -I/home/maps/vitis_ws/zynq_zed/export/zynq_zed/sw/standalone_ps7_cortexa9_0/include -Wall -Wextra      -O0  -g3     -U__clang__       CMakeFiles/zynq_zed_fib_app.elf.dir/platform.c.obj CMakeFiles/zynq_zed_fib_app.elf.dir/zynq_zed_fib_app.c.obj -o zynq_zed_fib_app.elf  -Wl,-T -Wl,"/home/maps/vitis_ws/zynq_zed_fib_app/src/lscript.ld" -L"/home/maps/vitis_ws/zynq_zed_fib_app/src/" -L"/home/maps/vitis_ws/zynq_zed/export/zynq_zed/sw/standalone_ps7_cortexa9_0/lib/" -L"/" -Wl,--start-group,-lxilstandalone -lxiltimer -lxil -lgcc -lc -Wl,--end-group 
arm-none-eabi-si

In [6]:
for app_name in app_names:
    elf_source_path = os.path.join(ws_path, app_name, "build", f"{app_name}.elf")
    elf_dest_path = os.path.join("bin", f"{app_name}.elf")
    os.makedirs("bin", exist_ok=True)
    if os.path.exists(elf_source_path):
        shutil.copy2(elf_source_path, elf_dest_path)
        print(f"Success! Compiled binary copied cleanly to: {elf_dest_path}")
    else:
        print(f"Compilation Warning: Executable not found at {elf_source_path}. Verify build logs.")

Success! Compiled binary copied cleanly to: bin/zynq_zed_leds_app.elf
Success! Compiled binary copied cleanly to: bin/zynq_zed_fib_app.elf


In [7]:
# Terminate the server connection and release the workspace lock
vitis.dispose()

Shutting down Vitis server running on port '42735'
